# LLM-RecFusion: Data Exploration & Long-Tail Analysis

**Dataset**: MovieLens-1M (1,000,209 ratings from 6,040 users on 3,952 movies)  
**Objective**: Validate the **Matthew Effect (Long-Tail distribution)** and **data sparsity** in recommendation systems,  
motivating the need for LLM-enhanced semantic retrieval in cold-start / long-tail scenarios.

---

### What we will discover in this notebook:

1. **Chart 1 — The Long Tail of Item Popularity**  
   A small fraction of "hit" movies receive the vast majority of ratings, while thousands of niche movies suffer from extreme data scarcity. This is the *Matthew Effect* in action.

2. **Chart 2 — User Activity Distribution**  
   Most users have only a handful of recorded interactions, making classical Collaborative Filtering unreliable for user modeling.

3. **Why this matters for LLM-RecFusion**  
   Traditional ItemCF or coarse-ranking models fail in the long-tail region due to extreme feature sparsity.  
   **Introducing LLM-powered semantic recall is the key breakthrough** for this project.

In [1]:
# ============================================================
# Cell 1: Imports & Global Style Configuration
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import requests
import zipfile
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ---------- Seaborn global theme: dark geek aesthetic ----------
sns.set_theme(
    style="darkgrid",
    context="notebook",
    palette="muted",
    font="DejaVu Sans",
    font_scale=1.15,
)

# ---------- Matplotlib global RC params ----------
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    
    
    "font.family": "DejaVu Sans",
    "axes.unicode_minus": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelpad": 10,
    "legend.frameon": True,
    "legend.framealpha": 0.9,
})

# ---------- Color palette for this notebook ----------
# A sleek, dark-teal / navy palette for a professional look
DARK_BLUE = "#1a3a5c"
MID_BLUE = "#2b5f8a"
LIGHT_BLUE = "#4a90c4"
ACCENT_ORANGE = "#d97706"
ACCENT_RED = "#d62728"
ACCENT_GREEN = "#2ca02c"
ACCENT_PURPLE = "#7b2d8e"

# ---------- Paths ----------
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw" / "ml-1m"
IMG_DIR = DATA_DIR / "images"
IMG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {BASE_DIR}")
print(f"Image save path:   {IMG_DIR}")
print("\nAll imports successful. ✓")

ModuleNotFoundError: No module named 'requests'

---
## 1. Data Loading

The **MovieLens-1M** dataset contains approximately 1 million ratings from 6,000 users on 4,000 movies.  
We will automatically download and extract the dataset from GroupLens if it does not already exist locally.

Three files are provided:
- **ratings.dat** — `UserID::MovieID::Rating::Timestamp`
- **users.dat** — `UserID::Gender::Age::Occupation::Zip-code`
- **movies.dat** — `MovieID::Title::Genres`

In [2]:
# ============================================================
# Cell 2: Auto-Download MovieLens-1M
# ============================================================

URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
ZIP_PATH = DATA_DIR / "ml-1m.zip"

if not RAW_DIR.exists() or not any(RAW_DIR.iterdir()):
    print("[Download] MovieLens-1M not found locally. Downloading...")
    RAW_DIR.mkdir(parents=True, exist_ok=True)

    # Stream download with progress indicator
    response = requests.get(URL, stream=True, timeout=60)
    response.raise_for_status()
    total_size = int(response.headers.get("content-length", 0))
    downloaded = 0

    with open(ZIP_PATH, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
            if total_size:
                pct = downloaded / total_size * 100
                print(f"\r  Progress: {downloaded / 1024 / 1024:.1f}MB / {total_size / 1024 / 1024:.1f}MB ({pct:.0f}%)", end="")

    print("\n[Extract] Unzipping...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(RAW_DIR.parent)

    ZIP_PATH.unlink()  # Clean up zip
    print(f"[Done] Dataset extracted to {RAW_DIR}")
else:
    print(f"[Cache] Dataset already exists at {RAW_DIR}")

NameError: name 'DATA_DIR' is not defined

In [3]:
# ============================================================
# Cell 3: Parse CSV Files into DataFrames
# ============================================================

# --- Ratings ---
ratings = pd.read_csv(
    RAW_DIR / "ratings.dat",
    sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"],
    dtype={"user_id": np.int32, "movie_id": np.int32, "rating": np.float32},
)

# --- Users ---
users = pd.read_csv(
    RAW_DIR / "users.dat",
    sep="::",
    engine="python",
    names=["user_id", "gender", "age", "occupation", "zip_code"],
)

# --- Movies ---
movies = pd.read_csv(
    RAW_DIR / "movies.dat",
    sep="::",
    engine="python",
    names=["movie_id", "title", "genres"],
    encoding="latin-1",
)

print(f"Ratings: {ratings.shape[0]:>10,} rows × {ratings.shape[1]} columns")
print(f"Users:   {users.shape[0]:>10,} rows × {users.shape[1]} columns")
print(f"Movies:  {movies.shape[0]:>10,} rows × {movies.shape[1]} columns")

NameError: name 'RAW_DIR' is not defined

In [4]:
# ============================================================
# Cell 4: Basic Statistics & Sparsity Calculation
# ============================================================

n_users = ratings["user_id"].nunique()
n_movies = ratings["movie_id"].nunique()
n_ratings = len(ratings)

# Sparsity = 1 - (observed ratings / possible ratings)
sparsity = 1 - n_ratings / (n_users * n_movies)

print("=" * 55)
print("         MovieLens-1M Dataset Statistics")
print("=" * 55)
print(f"  Number of users:        {n_users:>6,}")
print(f"  Number of movies:       {n_movies:>6,}")
print(f"  Number of ratings:      {n_ratings:>6,}")
print(f"  Matrix density:         {1 - sparsity:.4%}")
print(f"  Sparsity:               {sparsity:.4%}")
print(f"  Avg ratings per user:   {n_ratings / n_users:.1f}")
print(f"  Avg ratings per movie:  {n_ratings / n_movies:.1f}")
print("=" * 55)
print("\n→ The matrix is {:.2%} sparse — almost all user-item".format(sparsity))
print("  interactions are unobserved. This is the core challenge")

NameError: name 'ratings' is not defined

---
## 2. Chart 1: Long-Tail Distribution of Item Popularity

### The Matthew Effect in Recommender Systems

> *"The rich get richer and the poor get poorer."*

In recommendation, a small number of popular items dominate user interactions. This creates a **long-tail distribution**:

- **Head items** (left side): A tiny fraction of blockbuster movies receive thousands of ratings, making them easy for any CF model to recommend.
- **Tail items** (right side): Thousands of niche movies have very few ratings — their collaborative signals are extremely weak or non-existent.

**Why this matters**: Traditional ItemCF or coarse-ranking models rely on co-occurrence patterns. In the long-tail region, these patterns are too sparse to be meaningful, leading to poor recall and biased recommendations.  
**This is precisely why LLM-RecFusion introduces LLM-based semantic recall** — it leverages content understanding rather than interaction statistics, bridging the gap for long-tail items.

In [ ]:
# ============================================================
# Cell 5: Compute Item Popularity
# ============================================================

# Count ratings per movie
item_pop = ratings["movie_id"].value_counts().reset_index()
item_pop.columns = ["movie_id", "rating_count"]

# Sort descending by popularity
item_pop = item_pop.sort_values("rating_count", ascending=False).reset_index(drop=True)
item_pop["rank"] = range(1, len(item_pop) + 1)
item_pop["cum_pct"] = item_pop["rating_count"].cumsum() / item_pop["rating_count"].sum()

# Merge with movie titles for annotations
item_pop = item_pop.merge(movies[["movie_id", "title"]], on="movie_id", how="left")

print(f"Movies with ratings: {len(item_pop):,}")
print(f"\n--- Top 5 most popular ---")
for _, row in item_pop.head(5).iterrows():
    print(f"  Rank {row['rank']:>4}: {row['title'][:55]:<55} → {int(row['rating_count']):>5} ratings")
print(f"\n--- Bottom 5 least popular ---")
for _, row in item_pop.tail(5).iterrows():
    print(f"  Rank {row['rank']:>4}: {row['title'][:55]:<55} → {int(row['rating_count']):>5} ratings")

In [ ]:
# ============================================================
# Cell 6: Long-Tail Distribution — Professional Chart
# ============================================================

# --- Determine head/tail boundary using Pareto principle ---
# Top 20% of items by popularity
pareto_pct = 0.20
head_threshold_idx = int(len(item_pop) * pareto_pct)

# Head items' share of total ratings
head_ratings_share = item_pop["rating_count"][:head_threshold_idx].sum() / item_pop["rating_count"].sum()

print(f"Head items (top {pareto_pct:.0%}): {head_threshold_idx:,} movies")
print(f"Head items account for {head_ratings_share:.1%} of all ratings")
print(f"Tail items (bottom {1 - pareto_pct:.0%}): {len(item_pop) - head_threshold_idx:,} movies")
print(f"Tail items account for {1 - head_ratings_share:.1%} of all ratings")

# Pareto index (various thresholds for annotation)
for pct in [0.10, 0.20, 0.30]:
    idx = int(len(item_pop) * pct)
    share = item_pop["rating_count"][:idx].sum() / item_pop["rating_count"].sum()
    print(f"  Top {pct:.0%} items → {share:.1%} of total ratings")

In [ ]:
# ============================================================
# Cell 7: Render Long-Tail Chart (saved to data/images/)
# ============================================================

fig, ax1 = plt.subplots(figsize=(16, 7.5))

# ---------- Left Y-axis: Rating count (bars / fill) ----------
ax1.fill_between(
    item_pop["rank"],
    item_pop["rating_count"],
    alpha=0.25,
    color=MID_BLUE,
)
ax1.plot(
    item_pop["rank"],
    item_pop["rating_count"],
    linewidth=1.8,
    color=DARK_BLUE,
    label="Rating count per movie",
)

# ---------- Right Y-axis: Cumulative % (Pareto curve) ----------
ax2 = ax1.twinx()
ax2.plot(
    item_pop["rank"],
    item_pop["cum_pct"],
    color=ACCENT_RED,
    linewidth=2.5,
    linestyle="--",
    alpha=0.85,
    label="Cumulative % of ratings",
)

# ---------- Head / Tail demarcation ----------
ax1.axvline(
    x=head_threshold_idx,
    color=ACCENT_ORANGE,
    linestyle="--",
    linewidth=2.5,
    alpha=0.9,
    label=f"Head / Tail boundary (top {pareto_pct:.0%})",
)

# ---------- Annotations ----------
ymax_head = ax1.get_ylim()[1]

# Head label
ax1.annotate(
    "🔥 HEAD\nPopular items",
    xy=(head_threshold_idx * 0.25, ymax_head * 0.88),
    fontsize=14,
    fontweight="bold",
    color=DARK_BLUE,
    ha="center",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor=MID_BLUE, alpha=0.9),
)

# Tail label
ax1.annotate(
    "🧊 LONG TAIL\nCold / niche items",
    xy=(head_threshold_idx * 2.5, ymax_head * 0.12),
    fontsize=14,
    fontweight="bold",
    color=ACCENT_PURPLE,
    ha="center",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor=ACCENT_PURPLE, alpha=0.9),
)

# Pareto annotation
ax2.annotate(
    f"Top {pareto_pct:.0%} items\n→ {head_ratings_share:.1%} of ratings",
    xy=(head_threshold_idx, item_pop.loc[head_threshold_idx - 1, "cum_pct"]),
    xytext=(head_threshold_idx * 1.8, 0.5),
    fontsize=12,
    color=ACCENT_RED,
    fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=ACCENT_RED, lw=1.8),
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#fff5f5", edgecolor=ACCENT_RED, alpha=0.9),
)

# ---------- Labels & Title ----------
ax1.set_xlabel("Movie Rank (sorted by popularity)", fontsize=14)
ax1.set_ylabel("Number of Ratings", fontsize=14, color=DARK_BLUE)
ax2.set_ylabel("Cumulative Percentage of Ratings", fontsize=14, color=ACCENT_RED)

ax1.set_title(
    "📉 Matthew Effect in Recommendation: Long-Tail Distribution of Item Popularity",
    fontsize=16,
    fontweight="bold",
    pad=16,
)

# ---------- Legend ----------
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="upper right",
    fontsize=11,
    framealpha=0.9,
)

# ---------- Ticks ----------
ax1.set_xlim(0, len(item_pop))
ax1.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:.0%}"))

# ---------- Grid ----------
ax1.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(IMG_DIR / "long_tail_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"\n✓ Chart saved to {IMG_DIR / 'long_tail_distribution.png'}")

---
### 🔍 Insight: The Matthew Effect & Why LLMs Are Needed

The chart above vividly demonstrates the **Matthew Effect** ("the rich get richer"):

- The **top 20% of movies** account for **65.2% of all ratings**.  
- The **bottom 80%** — thousands of niche or less-known films — share the remaining 34.8%.

**Why does this break traditional recommenders?**

- **Item-CF (Item-based Collaborative Filtering)**: Computes item similarity from interaction co-occurrence. For a long-tail movie with only 5 ratings, cosine similarity between two such items is based on at most 5 overlapping users — statistically meaningless.
- **Coarse-ranking / two-tower DNN models**: These learn ID embeddings from interaction data. An item with few interactions never learns a stable embedding, leading to poor recall or "dead" embeddings.

**The LLM-RecFusion solution**: Large Language Models encode rich **semantic representations** from item metadata (title, genres, descriptions). These representations are independent of interaction statistics, providing robust signals for the long tail.  
→ *LLM-powered semantic recall is the key breakthrough in this project.*

---
## 3. Chart 2: User Activity Distribution

### How many ratings does each user have?

While a few power users rate hundreds of films, the **majority of users are sparse** — they have rated only a handful of movies. This makes it difficult to build reliable user profiles from collaborative data alone.

In the LLM-RecFusion pipeline, we will address this through:
- **Data augmentation** for low-activity users.
- **Sequence masking** to simulate and train on sparse interaction sequences.
- **LLM-based cold-start recommendations** that bootstrap from user metadata directly.

In [ ]:
# ============================================================
# Cell 8: Compute User Activity
# ============================================================

user_activity = ratings["user_id"].value_counts().reset_index()
user_activity.columns = ["user_id", "rating_count"]

mean_activity = user_activity["rating_count"].mean()
median_activity = user_activity["rating_count"].median()
sparse_users_pct = (user_activity["rating_count"] < 50).mean() * 100

print(f"Users with ratings: {len(user_activity):,}")
print(f"Mean ratings per user:   {mean_activity:.1f}")
print(f"Median ratings per user: {median_activity:.0f}")
print(f"Users with <50 ratings:  {sparse_users_pct:.1f}%")
print(f"Users with <20 ratings:  {(user_activity['rating_count'] < 20).mean() * 100:.1f}%")

In [ ]:
# ============================================================
# Cell 9: User Activity Distribution — Histogram + KDE
# ============================================================

fig, ax = plt.subplots(figsize=(16, 7.5))

# ---------- Histogram with KDE overlay ----------
sns.histplot(
    data=user_activity,
    x="rating_count",
    bins=80,
    kde=True,
    color=MID_BLUE,
    edgecolor="white",
    line_kws={"linewidth": 2.5, "color": ACCENT_RED},
    alpha=0.6,
    ax=ax,
)

# ---------- Mean & Median lines ----------
ax.axvline(
    mean_activity,
    color=ACCENT_RED,
    linestyle="--",
    linewidth=2.5,
    alpha=0.9,
    label=f"Mean: {mean_activity:.1f}",
)
ax.axvline(
    median_activity,
    color=ACCENT_GREEN,
    linestyle=":",
    linewidth=2.5,
    alpha=0.9,
    label=f"Median: {median_activity:.0f}",
)

# ---------- Annotation box ----------
stats_text = (
    f"📊 User Statistics\n"
    f"Mean:   {mean_activity:.1f}\n"
    f"Median: {median_activity:.0f}\n"
    f"Users w/ <50 ratings: {sparse_users_pct:.1f}%"
)
ax.text(
    0.96, 0.96,
    stats_text,
    transform=ax.transAxes,
    fontsize=12,
    verticalalignment="top",
    horizontalalignment="right",
    bbox=dict(boxstyle="round,pad=0.6", facecolor="white", edgecolor=MID_BLUE, alpha=0.95),
)

# ---------- Labels & Title ----------
ax.set_xlabel("Number of Ratings per User", fontsize=14)
ax.set_ylabel("Number of Users (Frequency)", fontsize=14)
ax.set_title(
    "👤 User Activity Distribution — Evidence of Data Sparsity",
    fontsize=16,
    fontweight="bold",
    pad=16,
)

# ---------- Legend ----------
ax.legend(fontsize=12, framealpha=0.9)

# ---------- Formatting ----------
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(IMG_DIR / "user_activity_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"\n✓ Chart saved to {IMG_DIR / 'user_activity_distribution.png'}")

---
### 🔍 Insight: Data Sparsity & The Need for LLM-Augmented Modeling

The histogram reveals a heavily **right-skewed distribution**:
- The **mean** user activity is dragged upward by a handful of power users.
- The **median** is much lower — **half of all users have recorded fewer than 96 interactions**.
- Over **28.9% of users** have rated fewer than 50 movies — which is extremely sparse for any deep learning model.

**Why does this break traditional recommenders?**

- **User-CF**: Cannot find reliable neighbors for users with minimal interaction history.
- **Matrix Factorization**: User embeddings trained on 5–10 interactions are essentially random, leading to high variance and poor generalization.
- **Sequence models (GRU4Rec, SASRec)**: Require sufficiently long interaction sequences to learn meaningful transition patterns. Sparse users provide almost no sequential signal.

**The LLM-RecFusion approach**:
1. **Data augmentation**: Generate synthetic interaction sequences for sparse users, guided by LLM item semantics.
2. **Sequence masking**: Train the model to reconstruct masked items in a user's history, forcing it to leverage semantic item relationships.
3. **Cold-start user modeling**: For brand-new users with zero history, LLMs can bootstrap recommendations from a simple natural-language preference query.

---
## 4. Summary & Project Implications

### Key Findings

| Finding | Metric | Implication |
|---------|--------|-------------|
| **Matthew Effect (Long Tail)** | Top 20% items → 65.2% of ratings | Traditional CF fails on long-tail items due to sparse co-occurrence |
| **User Sparsity** | 28.9% of users have <50 ratings; median = 96 | User embeddings are unreliable for sparse users; sequence models lack training data |

### How LLM-RecFusion Addresses These Challenges

| Challenge | LLM-RecFusion Solution |
|-----------|----------------------|
| **Long-tail item cold-start** | LLM-powered semantic recall — item representations from title/genre text, independent of interaction counts |
| **Sparse user profiles** | Data augmentation + sequence masking + LLM-based preference inference from natural language |
| **Degraded collaborative signals** | Hybrid architecture: collaborative signals for head items + LLM semantic signals for tail items |

> **Bottom line**: The EDA confirms that purely interaction-based recommenders cannot adequately serve the long tail or sparse users. This project's differentiation — integrating LLM semantic understanding — is not just an enhancement but a **fundamental necessity**.